# Training regimes
Saved evidence only. No training or network access.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
study = json.loads(Path(study_path).read_text())
print(study['status'], study['limits'])
rows = [r for r in study['measurements'] if r['opponent'] == 'random-smoke-anchor' and r.get('phase', 'development') == 'development']
fig, ax = plt.subplots(figsize=(8,4))
for name, seed, variant in sorted({(r['regime'], r['seed'], r.get('variant', 'raw')) for r in rows}):
    points = sorted([r for r in rows if (r['regime'], r['seed'], r.get('variant', 'raw')) == (name, seed, variant) and r['complete'] and r['score'] is not None], key=lambda r: r['training_seconds'])
    ax.step([r['training_seconds']/3600 for r in points], [r['score'] for r in points], where='post', marker='o', label=f'{name} seed {seed} {variant}')
ax.set(xlabel='Actual cumulative training hours', ylabel='Score vs fixed random anchor', ylim=(0,1))
if rows: ax.legend()
fig.tight_layout()
fig.savefig('learning-curves.png')
plt.show()
print('Training seeds:', study.get('seeds', [study.get('seed')]), '; small cohorts are descriptive only.')
for item in study['runs']:
    if not Path(item['path']).exists():
        print(item, 'Run export unavailable: failure before persistence')
        continue
    run = json.loads(Path(item['path']).read_text())
    for stage in run['stages']:
        print(run['regime']['id'], stage['id'], stage['status'], stage['diagnostics'])


In [ ]:
comparison = json.loads((Path(study_path).parent / 'cost-comparison.json').read_text())
print('Common observed cost window:', comparison)
if comparison['status'] == 'available' and study['study'] == 'ataraxos-ablations':
    points = comparison['rows']
    controls = {(p['seed'], p['variant']): p['score'] for p in points if p['regime'] == 'rl-control'}
    fig, ax = plt.subplots(figsize=(9,4))
    ax.scatter([p['regime'] for p in points], [p['score'] - controls[(p['seed'], p['variant'])] for p in points])
    ax.axhline(0, color='gray')
    ax.set(ylabel='Score difference vs control at shared cost', title='Paired seed effects: descriptive evidence only')
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig('ablation-effects.png')
    plt.show()
print('S1-S5: unavailable; current-world scenario premises not yet certified.')
print('Raw/EMA: distinct artifact identities, correlated outputs within a training seed; never additional replicates.')


In [ ]:
for name in ('anchor-cost-comparisons.json', 'cost-cutoffs.json', 'uncertainty.json'):
    print(name, json.loads((Path(study_path).parent / name).read_text()))
print('Development and endpoint cohorts are separate; checkpoint-index matches do not imply equal cost.')
